# SeaAlert - Notebook 01: Generate Synthetic Dataset

This notebook generates a synthetic maritime message dataset for severity classification.

**Labels:** Routine, Safety, Urgency, Distress  
**Styles:** formal, informal, third_party  
**Scenarios:** water_ingress, engine_failure, medical_issue, fire_smoke, person_overboard, nav_hazard, tow_request, radio_check, etc.

## Cell 0 - Setup

In [6]:
# Mount Google Drive (Colab)
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# Install dependencies
!pip install -q pandas numpy tqdm openai jsonschema

# Imports
import os
import random
import json
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm import tqdm

# Seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Project directory
if IN_COLAB:
    PROJECT_DIR = Path("/content/drive/MyDrive/SeaAlert")
else:
    PROJECT_DIR = Path(".").resolve().parent

# Create folders
DATA_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results"
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Add src to path
sys.path.insert(0, str(PROJECT_DIR / "src"))

# Load API key
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

# Load API key - try from file first, then environment variable
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
try:
    from API_KEY import OPENAI_API_KEY as FILE_API_KEY
    if FILE_API_KEY:
        OPENAI_API_KEY = FILE_API_KEY
        print("API key loaded from src/API_KEY.py")
except ImportError:
    print("API_KEY.py not found, using environment variable")

# Quick run mode (no LLM, use templates)
QUICK_RUN = False  # Set True for debugging

if not QUICK_RUN and not OPENAI_API_KEY:
    print("WARNING: OPENAI_API_KEY not set. Set QUICK_RUN=True for template-based generation.")

print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"QUICK_RUN: {QUICK_RUN}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
API key loaded from src/API_KEY.py
PROJECT_DIR: /content/drive/MyDrive/SeaAlert
QUICK_RUN: False


## Cell 1 - Configuration

In [7]:
# Dataset configuration
N_TOTAL = 2000 if not QUICK_RUN else 80  # Total samples to generate

LABELS = ["Routine", "Safety", "Urgency", "Distress"]
STYLES = ["formal", "informal", "third_party"]
SCENARIOS = [
    "water_ingress", "engine_failure", "medical_issue", "fire_smoke",
    "person_overboard", "nav_hazard", "tow_request", "radio_check",
    "grounding", "collision", "cargo_shift", "steering_failure"
]

# Codeword settings
CODEWORDS = {
    "Distress": "MAYDAY",
    "Urgency": "PAN PAN",
    "Safety": "SECURITE",
    "Routine": "NONE"
}
TARGET_CODEWORD_RATE = 0.5  # Target rate of samples with codewords

# Generation settings
MAX_RETRIES = 3
BATCH_SAVE_EVERY = 50

# Output paths
DATASET_PATH = DATA_DIR / "02seaalert.csv"
SPLIT_PATH = RESULTS_DIR / "split_indices.csv"
QUALITY_REPORT_PATH = RESULTS_DIR / "data_quality_report.txt"
CHECKPOINT_PATH = DATA_DIR / "tmp_partial.csv"

print(f"Target samples: {N_TOTAL}")
print(f"Labels: {LABELS}")
print(f"Styles: {STYLES}")
print(f"Scenarios: {SCENARIOS}")

Target samples: 2000
Labels: ['Routine', 'Safety', 'Urgency', 'Distress']
Styles: ['formal', 'informal', 'third_party']
Scenarios: ['water_ingress', 'engine_failure', 'medical_issue', 'fire_smoke', 'person_overboard', 'nav_hazard', 'tow_request', 'radio_check', 'grounding', 'collision', 'cargo_shift', 'steering_failure']


## Cell 2 - Prompt + Schema

In [8]:
# JSON Schema for generated samples
SAMPLE_SCHEMA = {
    "type": "object",
    "required": ["text", "label", "style", "scenario_type", "has_codeword", "codeword"],
    "properties": {
        "text": {"type": "string", "minLength": 20},
        "label": {"type": "string", "enum": LABELS},
        "style": {"type": "string", "enum": STYLES},
        "scenario_type": {"type": "string"},
        "has_codeword": {"type": "boolean"},
        "codeword": {"type": "string", "enum": ["MAYDAY", "PAN PAN", "SECURITE", "NONE"]},
        "vessel": {"type": ["string", "null"]},
        "call_sign": {"type": ["string", "null"]},
        "mmsi": {"type": ["string", "null"]},
        "location": {"type": ["string", "null"]},
        "weather": {"type": ["string", "null"]},
        "pob": {"type": ["string", "integer", "null"]},
        "nature": {"type": ["string", "null"]},
        "injury": {"type": ["string", "null"]}
    }
}

def build_generation_prompt(label, style, scenario, include_codeword=True, make_tricky=False):
    """Build LLM prompt for generating a maritime message."""
    codeword = CODEWORDS.get(label, "NONE")

    style_instructions = {
        "formal": "Use formal GMDSS protocol structure with proper maritime terminology.",
        "informal": "Use casual radio speech with fillers, hesitations, incomplete sentences.",
        "third_party": "Write as a relay or report from another vessel or station."
    }

    tricky_instructions = ""
    if make_tricky:
        tricky_options = [
            "Include a negation that changes meaning (e.g., 'NOT a distress', 'no longer').",
            "Mention a drill or test scenario.",
            "Use keywords that could mislead (e.g., 'fire' in a routine context).",
            "Include a cancellation or all-clear message.",
            "Reference a past resolved incident.",
        ]
        tricky_instructions = f"\nSPECIAL: {random.choice(tricky_options)}"

    codeword_instruction = ""
    if include_codeword and label != "Routine":
        codeword_instruction = f"Include the codeword '{codeword}' appropriately."
    elif not include_codeword:
        codeword_instruction = "Do NOT include any GMDSS codewords (MAYDAY, PAN PAN, SECURITE)."

    prompt = f"""Generate a realistic maritime VHF/GMDSS radio message with these requirements:

LABEL: {label}
STYLE: {style} - {style_instructions[style]}
SCENARIO: {scenario}
{codeword_instruction}
{tricky_instructions}

REQUIREMENTS:
1. Create unique, realistic maritime communication
2. Vary sentence structure and information order
3. Include realistic details: vessel names, positions, conditions
4. Avoid template-like repetitive phrases
5. Make it sound like authentic radio communication
6. Length should vary (20-150 words depending on urgency)

Return ONLY valid JSON with this exact structure:
{{
    "text": "The actual radio message text",
    "label": "{label}",
    "style": "{style}",
    "scenario_type": "{scenario}",
    "has_codeword": true/false,
    "codeword": "MAYDAY|PAN PAN|SECURITE|NONE",
    "vessel": "vessel name or null",
    "call_sign": "call sign or null",
    "mmsi": "MMSI number or null",
    "location": "position/location or null",
    "weather": "weather conditions or null",
    "pob": "persons on board (number) or null",
    "nature": "nature of incident or null",
    "injury": "injury details or null"
}}"""

    return prompt

def build_repair_prompt(malformed_text, error):
    """Build prompt to repair malformed JSON."""
    return f"""The following JSON is malformed. Fix it and return valid JSON only.

ERROR: {error}

MALFORMED:
{malformed_text}

Return ONLY the corrected valid JSON, nothing else."""

# Test prompt
test_prompt = build_generation_prompt("Distress", "formal", "fire_smoke", True, False)
print("Sample prompt:")
print(test_prompt[:500] + "...")

Sample prompt:
Generate a realistic maritime VHF/GMDSS radio message with these requirements:

LABEL: Distress
STYLE: formal - Use formal GMDSS protocol structure with proper maritime terminology.
SCENARIO: fire_smoke
Include the codeword 'MAYDAY' appropriately.


REQUIREMENTS:
1. Create unique, realistic maritime communication
2. Vary sentence structure and information order
3. Include realistic details: vessel names, positions, conditions
4. Avoid template-like repetitive phrases
5. Make it sound like authen...


## Cell 3 - LLM Generation Engine

In [ ]:
def detect_codeword_in_text(text):
    """
    Detects if any standard GMDSS codeword (MAYDAY, PAN PAN, SECURITE) 
    is present in the given text.
    
    Args:
        text (str): The maritime message to check.
        
    Returns:
        tuple: (bool, str) - True if found, and the codeword itself (or "NONE").
    """
    text_upper = text.upper()
    
    # Priority check: MAYDAY is most critical
    if "MAYDAY" in text_upper:
        return True, "MAYDAY"
    # Handling variations of PAN PAN
    elif "PAN PAN" in text_upper or "PAN-PAN" in text_upper:
        return True, "PAN PAN"
    # Handling SECURITE (sometimes misspelled or repeated)
    elif "SECURITE" in text_upper:
        return True, "SECURITE"
        
    return False, "NONE"

def validate_sample(data):
    """
    Validates a single generated sample to ensure it meets all structural requirements.
    
    Checks for:
    1. Presence of all required fields.
    2. Validity of categorical values (Label, Style).
    3. Minimum text length to ensure content quality.
    
    Args:
        data (dict): The generated sample data.
        
    Returns:
        tuple: (bool, str) - True if valid, otherwise False and error message.
    """
    required = ["text", "label", "style", "scenario_type", "has_codeword", "codeword"]
    
    # Check for missing keys
    for field in required:
        if field not in data:
            return False, f"Missing field: {field}"
            
    # Validate categorical constraints
    if data["label"] not in LABELS:
        return False, f"Invalid label: {data['label']}"
    if data["style"] not in STYLES:
        return False, f"Invalid style: {data['style']}"
        
    # Ensure text is not empty or too short (e.g., failed generation)
    if not isinstance(data["text"], str) or len(data["text"]) < 20:
        return False, "Text too short"
        
    return True, ""

def call_llm(prompt, max_retries=3):
    """
    Executes a call to the OpenAI API with retry logic for robustness.
    
    Uses 'gpt-4o-mini' to generate synthetic maritime messages.
    
    Args:
        prompt (str): The prompt instructions for the LLM.
        max_retries (int): Number of attempts in case of API failure.
        
    Returns:
        str: The raw text response from the LLM, or None if all retries failed.
    """
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": "You are a maritime communications expert."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.8, # Higher temperature for diverse content generation
                max_tokens=500,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            if attempt == max_retries - 1:
                print(f"LLM call failed after {max_retries} attempts: {e}")
                return None
    return None

def parse_llm_response(response):
    """
    Parses the raw string response from the LLM into a structured JSON object.
    
    Handles common LLM formatting issues (e.g., Markdown code blocks).
    
    Args:
        response (str): The raw string output from the LLM.
        
    Returns:
        dict: Parsed JSON object, or None if parsing fails.
    """
    if not response:
        return None
        
    response = response.strip()
    
    # Clean up Markdown code blocks (```json ... ```)
    if response.startswith("```"):
        lines = response.split("\n")
        # Remove the first and last lines (the backticks)
        lines = [l for l in lines if not l.strip().startswith("```")]
        response = "\n".join(lines)
        
    try:
        return json.loads(response)
    except:
        # Fallback: Try to extract JSON structure using regex if extra text exists
        match = re.search(r'\{.*\}', response, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except:
                pass
    return None

def create_generation_plan(n_total, codeword_rate=0.5, tricky_rate=0.2):
    """
    Creates a balanced generation plan to ensure equal representation across classes.
    
    The plan distributes samples evenly across Labels, Styles, and Scenarios.
    
    Args:
        n_total (int): Total number of samples to generate.
        codeword_rate (float): Probability of including a distress codeword (excluding Routine).
        tricky_rate (float): Probability of generating an adversarial/tricky sample.
        
    Returns:
        list: A list of dictionaries, where each dict is a blueprint for one sample.
    """
    plan = []
    
    # Calculate target samples per category to ensure perfect balance
    samples_per_label = n_total // len(LABELS)
    samples_per_combo = max(1, samples_per_label // (len(STYLES) * len(SCENARIOS)))

    for label in LABELS:
        for style in STYLES:
            for scenario in SCENARIOS:
                for _ in range(samples_per_combo):
                    # Decisions for this specific sample
                    include_cw = random.random() < codeword_rate
                    # Routine messages rarely/never use distress codewords
                    if label == "Routine":
                        include_cw = False
                        
                    make_tricky = random.random() < tricky_rate
                    
                    plan.append({
                        "label": label,
                        "style": style,
                        "scenario": scenario,
                        "include_codeword": include_cw,
                        "make_tricky": make_tricky,
                    })
    
    # Shuffle to ensure random order during generation
    random.shuffle(plan)
    return plan[:n_total]

# Output plan size summary
print(f"Generation plan prepared. Total samples aimed: {len(create_generation_plan(N_TOTAL))}")

Generation plan size: 1872


In [10]:
# Quick run: Template-based generation (no LLM needed)
def generate_dataset_quick(n_total=80):
    """Generate dataset using templates (for testing without LLM)."""
    templates = {
        "Distress": [
            "MAYDAY MAYDAY MAYDAY. This is {vessel}, {call_sign}. We are taking on water at position {location}. {pob} persons on board. Require immediate assistance.",
            "MAYDAY. Motor vessel {vessel}. Fire in engine room. Position {location}. Preparing to abandon ship. {pob} POB.",
            "{vessel} calling all stations. MAYDAY. {nature} at {location}. Request immediate help.",
            "All stations, all stations. MAYDAY relay. {vessel} reports {nature}. Position {location}. Immediate assistance required.",
        ],
        "Urgency": [
            "PAN PAN PAN PAN PAN PAN. All stations. This is {vessel}. We have {nature} on board. Position {location}. Require medical advice.",
            "PAN PAN. {vessel}. Engine failure. Drifting at {location}. Request tow assistance.",
            "All stations this is {vessel}. PAN PAN. Person overboard. Position {location}. Conducting search.",
            "PAN PAN. {vessel} requesting urgent medical evacuation at {location}.",
        ],
        "Safety": [
            "SECURITE SECURITE SECURITE. All stations. Navigation hazard reported at {location}. Container adrift.",
            "SECURITE. This is Coast Guard. Weather warning for area {location}. Gale force winds expected.",
            "{vessel} to all vessels. SECURITE. Fishing nets in water near {location}.",
            "SECURITE. All vessels. Unlit buoy reported at {location}. Navigate with caution.",
        ],
        "Routine": [
            "Radio check, radio check. This is {vessel}, {call_sign}. How do you read?",
            "{vessel} departing {location} bound for port. ETA 1400. All well on board.",
            "All stations, this is {vessel}. Routine traffic. Position report at {location}.",
            "{vessel} to Port Control. Requesting berth assignment. Arrival in 2 hours.",
        ],
    }

    vessels = ["Blue Horizon", "Sea Eagle", "Northern Star", "Pacific Dawn",
               "Ocean Spirit", "Atlantic Voyager", "Seafarer", "Viking Queen"]
    positions = ["47.32N 008.12W", "52.15N 004.30E", "41.50N 012.45E",
                 "38.20N 009.10W", "35.40N 014.25E"]
    natures = ["medical emergency", "engine failure", "fire", "water ingress",
               "steering failure", "crew injury"]

    samples = []
    plan = create_generation_plan(n_total)

    for idx, params in enumerate(tqdm(plan, desc="Generating samples")):
        label = params["label"]
        style = params["style"]
        scenario = params["scenario"]

        template = random.choice(templates[label])
        vessel = random.choice(vessels)
        call_sign = ''.join(random.choices('ABCDEFGHIJKLMNOPQRSTUVWXYZ', k=4)) + str(random.randint(1,9))

        text = template.format(
            vessel=vessel,
            call_sign=call_sign,
            location=random.choice(positions),
            pob=random.randint(2, 25),
            nature=random.choice(natures),
        )

        # Style modifications
        if style == "informal":
            fillers = ["uh", "um", "you know", "like", "we got"]
            words = text.split()
            if len(words) > 5:
                insert_pos = random.randint(3, len(words) - 1)
                words.insert(insert_pos, random.choice(fillers))
            text = " ".join(words)
        elif style == "third_party":
            text = f"Relay message: Received report from another vessel. {text}"

        has_cw, codeword = detect_codeword_in_text(text)

        samples.append({
            "idx": idx,
            "text": text,
            "label": label,
            "style": style,
            "scenario_type": scenario,
            "has_codeword": has_cw,
            "codeword": codeword,
            "vessel": vessel,
            "call_sign": call_sign,
            "mmsi": None,
            "location": random.choice(positions),
            "weather": None,
            "pob": random.randint(2, 15),
            "nature": random.choice(natures) if label != "Routine" else None,
            "injury": None,
        })

    return pd.DataFrame(samples)

# LLM-based generation
def generate_dataset_llm(n_total):
    """Generate dataset using LLM."""
    samples = []

    # Load checkpoint if exists
    if CHECKPOINT_PATH.exists():
        existing = pd.read_csv(CHECKPOINT_PATH)
        samples = existing.to_dict('records')
        print(f"Resuming from checkpoint: {len(samples)} samples")

    plan = create_generation_plan(n_total)
    remaining = plan[len(samples):]

    pbar = tqdm(remaining, desc=f"Generating ({len(samples)}/{n_total})")

    for params in pbar:
        prompt = build_generation_prompt(
            label=params["label"],
            style=params["style"],
            scenario=params["scenario"],
            include_codeword=params["include_codeword"],
            make_tricky=params["make_tricky"],
        )

        response = call_llm(prompt)
        if response is None:
            continue

        data = parse_llm_response(response)
        if data is None:
            repair_prompt = build_repair_prompt(response, "JSON parse error")
            repair_response = call_llm(repair_prompt, max_retries=1)
            data = parse_llm_response(repair_response)

        if data is None:
            continue

        is_valid, error = validate_sample(data)
        if not is_valid:
            continue

        # Reconcile codewords
        detected_has, detected_cw = detect_codeword_in_text(data["text"])
        data["has_codeword"] = detected_has
        data["codeword"] = detected_cw if detected_has else "NONE"
        data["idx"] = len(samples)

        samples.append(data)
        pbar.set_description(f"Generated {len(samples)}/{n_total}")

        # Checkpoint
        if len(samples) % BATCH_SAVE_EVERY == 0:
            pd.DataFrame(samples).to_csv(CHECKPOINT_PATH, index=False)

    return pd.DataFrame(samples)

# Run generation
if QUICK_RUN:
    print("Running in QUICK_RUN mode (template-based)")
    df = generate_dataset_quick(N_TOTAL)
else:
    print("Running with LLM generation")
    df = generate_dataset_llm(N_TOTAL)

print(f"\nGenerated {len(df)} samples")
df.head()

Running with LLM generation


Generated 1872/2000: 100%|██████████| 1872/1872 [2:40:06<00:00,  5.13s/it]


Generated 1872 samples


,text,label,style,scenario_type,has_codeword,codeword,vessel,call_sign,mmsi,location,weather,pob,nature,injury,idx
0,"MAYDAY, MAYDAY, MAYDAY. This is the fishing ve...",Distress,formal,nav_hazard,True,MAYDAY,Ocean Hunter,WXYZ123,123456789,"37 degrees 45 minutes North, 122 degrees 30 mi...",overcast with a light swell,6,navigation hazard,no injuries reported,0
1,"This is the vessel Ocean Explorer, call sign A...",Safety,formal,medical_issue,False,NONE,Ocean Explorer,ABCD123,123456789,"34 degrees 15 minutes North, 120 degrees 45 mi...",clear with calm seas,15,medical issue,severe laceration to the arm,1
2,"SECURITE, SECURITE. This is the fishing vessel...",Safety,third_party,steering_failure,True,SECURITE,Ocean's Bounty,WZ1234,123456789,"34°12.5'N, 118°28.9'W","Calm, 5 knots from the west",5,steering failure,none,2
3,"This is the vessel Ocean Explorer, call sign A...",Distress,formal,medical_issue,False,NONE,Ocean Explorer,ABC123,123456789,"34 degrees 15 minutes North, 118 degrees 30 mi...",calm with clear visibility,12,severe allergic reaction,critical condition,3
4,"This is the fishing vessel Ocean Dawn, current...",Distress,formal,water_ingress,False,NONE,Ocean Dawn,WDC1234,123456789,"34 degrees 15 minutes North, 75 degrees 30 min...","Sea conditions rough, swells 3 meters, winds 3...",5,water ingress in engine room,None,4


## Cell 4 - Masking + Derived Columns

In [11]:
def mask_codewords(text, replacement="[SIGNAL]"):
    """Replace codewords with placeholder."""
    result = re.sub(r'\bMAYDAY\b', replacement, text, flags=re.IGNORECASE)
    result = re.sub(r'\bPAN\s*PAN\b', replacement, result, flags=re.IGNORECASE)
    result = re.sub(r'\bSECURITE\b', replacement, result, flags=re.IGNORECASE)
    return result

# Add text_masked column
df["text_masked"] = df["text"].apply(mask_codewords)

# Verify has_codeword and codeword columns
for idx, row in df.iterrows():
    detected_has, detected_cw = detect_codeword_in_text(row["text"])
    df.at[idx, "has_codeword"] = detected_has
    df.at[idx, "codeword"] = detected_cw

# Ensure idx is sequential
df["idx"] = range(len(df))

print("Columns:", df.columns.tolist())
print("\nSample with masking:")
sample = df[df["has_codeword"] == True].iloc[0] if df["has_codeword"].any() else df.iloc[0]
print(f"Original: {sample['text'][:100]}...")
print(f"Masked: {sample['text_masked'][:100]}...")

Columns: ['text', 'label', 'style', 'scenario_type', 'has_codeword', 'codeword', 'vessel', 'call_sign', 'mmsi', 'location', 'weather', 'pob', 'nature', 'injury', 'idx', 'text_masked']

Sample with masking:
Original: MAYDAY, MAYDAY, MAYDAY. This is the fishing vessel Ocean Hunter, call sign WXYZ123, MMSI 123456789. ...
Masked: [SIGNAL], [SIGNAL], [SIGNAL]. This is the fishing vessel Ocean Hunter, call sign WXYZ123, MMSI 12345...


## Cell 5 - Data Quality Checks

In [12]:
report_lines = []
report_lines.append("=" * 60)
report_lines.append("DATA QUALITY REPORT")
report_lines.append("=" * 60)
report_lines.append(f"\nTotal samples: {len(df)}")

# Missing values
report_lines.append("\n--- Missing Values ---")
for col in df.columns:
    missing = df[col].isna().sum()
    if missing > 0:
        report_lines.append(f"  {col}: {missing} ({100*missing/len(df):.1f}%)")

# Label distribution
report_lines.append("\n--- Label Distribution ---")
label_counts = df["label"].value_counts()
for label, count in label_counts.items():
    report_lines.append(f"  {label}: {count} ({100*count/len(df):.1f}%)")

# Style distribution
report_lines.append("\n--- Style Distribution ---")
style_counts = df["style"].value_counts()
for style, count in style_counts.items():
    report_lines.append(f"  {style}: {count} ({100*count/len(df):.1f}%)")

# Scenario distribution
report_lines.append("\n--- Scenario Distribution ---")
scenario_counts = df["scenario_type"].value_counts()
for scenario, count in scenario_counts.head(8).items():
    report_lines.append(f"  {scenario}: {count}")

# Codeword distribution
report_lines.append("\n--- Codeword Distribution ---")
cw_counts = df["has_codeword"].value_counts()
report_lines.append(f"  With codeword: {cw_counts.get(True, 0)}")
report_lines.append(f"  Without codeword: {cw_counts.get(False, 0)}")

codeword_types = df["codeword"].value_counts()
for cw, count in codeword_types.items():
    report_lines.append(f"    {cw}: {count}")

# Text length stats
report_lines.append("\n--- Text Length Statistics ---")
word_counts = df["text"].apply(lambda x: len(str(x).split()))
char_counts = df["text"].apply(lambda x: len(str(x)))
report_lines.append(f"  Words: min={word_counts.min()}, max={word_counts.max()}, mean={word_counts.mean():.1f}")
report_lines.append(f"  Chars: min={char_counts.min()}, max={char_counts.max()}, mean={char_counts.mean():.1f}")

# Duplicate check
report_lines.append("\n--- Duplicate Check ---")
exact_dups = df["text"].duplicated().sum()
report_lines.append(f"  Exact duplicates: {exact_dups}")

# Normalized duplicate check
def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = ' '.join(text.split())
    return text

df["text_normalized"] = df["text"].apply(normalize_text)
near_dups = df["text_normalized"].duplicated().sum()
report_lines.append(f"  Near-duplicates (normalized): {near_dups}")

# Masking verification
report_lines.append("\n--- Masking Verification ---")
masked_still_has_cw = df["text_masked"].apply(lambda x: bool(re.search(r'\b(MAYDAY|PAN\s*PAN|SECURITE)\b', str(x), re.IGNORECASE))).sum()
report_lines.append(f"  Masked texts still containing codewords: {masked_still_has_cw}")

# Print report
report_text = "\n".join(report_lines)
print(report_text)

# Save report
with open(QUALITY_REPORT_PATH, 'w') as f:
    f.write(report_text)
print(f"\nReport saved to: {QUALITY_REPORT_PATH}")

# Drop temporary column
df = df.drop(columns=["text_normalized"])

DATA QUALITY REPORT

Total samples: 1872

--- Missing Values ---
  call_sign: 191 (10.2%)
  mmsi: 315 (16.8%)
  weather: 4 (0.2%)
  pob: 110 (5.9%)
  nature: 82 (4.4%)
  injury: 432 (23.1%)

--- Label Distribution ---
  Distress: 468 (25.0%)
  Safety: 468 (25.0%)
  Routine: 468 (25.0%)
  Urgency: 468 (25.0%)

--- Style Distribution ---
  formal: 624 (33.3%)
  third_party: 624 (33.3%)
  informal: 624 (33.3%)

--- Scenario Distribution ---
  nav_hazard: 156
  medical_issue: 156
  steering_failure: 156
  water_ingress: 156
  radio_check: 156
  fire_smoke: 156
  cargo_shift: 156
  tow_request: 156

--- Codeword Distribution ---
  With codeword: 655
  Without codeword: 1217
    NONE: 1217
    SECURITE: 237
    PAN PAN: 212
    MAYDAY: 206

--- Text Length Statistics ---
  Words: min=35, max=129, mean=79.0
  Chars: min=212, max=766, mean=463.1

--- Duplicate Check ---
  Exact duplicates: 0
  Near-duplicates (normalized): 0

--- Masking Verification ---
  Masked texts still containing codewor

In [13]:
# Show sample examples
print("\n" + "=" * 60)
print("SAMPLE EXAMPLES")
print("=" * 60)

for label in LABELS:
    print(f"\n--- {label} ---")
    label_samples = df[df["label"] == label]
    for style in STYLES:
        style_samples = label_samples[label_samples["style"] == style]
        if len(style_samples) > 0:
            sample = style_samples.sample(min(2, len(style_samples)), random_state=SEED)
            for _, row in sample.iterrows():
                print(f"  [{style}] {row['text'][:120]}...")


SAMPLE EXAMPLES

--- Routine ---
  [formal] This is the vessel Ocean Voyager, call sign ABC123, reporting a person overboard incident. We are located at latitude 34...
  [formal] This is the fishing vessel Ocean Voyager, call sign 2GHI5, MMSI 123456789. We are currently grounded at position 34 degr...
  [informal] Hey there, all vessels in the area, this is the S.S. Ocean Breeze. Just wanted to give a heads up about some floating de...
  [informal] Uh, this is the fishing boat Sea Breeze, uh, we’ve had a person go overboard, I repeat, overboard. Uh, last saw him, lik...
  [third_party] This is a report from the bulk carrier 'Ocean Star', currently anchored at position 35 degrees 12 minutes North, 75 degr...
  [third_party] Attention all vessels, this is the fishing trawler Green Wave relaying a message from the cargo ship Ocean Star. The Oce...

--- Safety ---
  [formal] Attention all vessels, this is the fishing trawler Ocean Explorer, call sign ABC123. Our current position is 37 deg

## Cell 6 - Reproducible Splits

In [14]:
from sklearn.model_selection import train_test_split

# Stratified split by label
train_val_idx, test_idx = train_test_split(
    df["idx"].values,
    test_size=0.15,
    stratify=df["label"],
    random_state=SEED
)

train_val_df = df[df["idx"].isin(train_val_idx)]

train_idx, val_idx = train_test_split(
    train_val_df["idx"].values,
    test_size=0.15/0.85,  # ~15% of total
    stratify=train_val_df["label"],
    random_state=SEED
)

# Create split dataframe
split_data = []
for idx in df["idx"].values:
    if idx in train_idx:
        split = "train"
    elif idx in val_idx:
        split = "val"
    else:
        split = "test"
    split_data.append({"idx": idx, "split": split})

split_df = pd.DataFrame(split_data)

# Verify distributions
print("Split sizes:")
print(split_df["split"].value_counts())

print("\nLabel distribution by split:")
for split in ["train", "val", "test"]:
    split_indices = split_df[split_df["split"] == split]["idx"].values
    split_labels = df[df["idx"].isin(split_indices)]["label"]
    print(f"\n{split}:")
    print(split_labels.value_counts(normalize=True).round(3))

# Save splits
split_df.to_csv(SPLIT_PATH, index=False)
print(f"\nSplit indices saved to: {SPLIT_PATH}")

Split sizes:
split
train    1310
test      281
val       281
Name: count, dtype: int64

Label distribution by split:

train:
label
Distress    0.25
Routine     0.25
Urgency     0.25
Safety      0.25
Name: proportion, dtype: float64

val:
label
Urgency     0.253
Safety      0.249
Distress    0.249
Routine     0.249
Name: proportion, dtype: float64

test:
label
Safety      0.253
Distress    0.249
Routine     0.249
Urgency     0.249
Name: proportion, dtype: float64

Split indices saved to: /content/drive/MyDrive/SeaAlert/results/split_indices.csv


## Cell 7 - Save Final Dataset

In [15]:
# Ensure correct column order
column_order = [
    "idx", "text", "label", "style", "scenario_type",
    "has_codeword", "codeword", "text_masked",
    "vessel", "call_sign", "mmsi", "location", "weather", "pob", "nature", "injury"
]

# Keep only columns that exist
final_columns = [c for c in column_order if c in df.columns]
df_final = df[final_columns].copy()

# Save dataset
df_final.to_csv(DATASET_PATH, index=False)
print(f"Dataset saved to: {DATASET_PATH}")
print(f"Shape: {df_final.shape}")
print(f"\nColumns: {df_final.columns.tolist()}")

# Clean up checkpoint
if CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
    print(f"\nCheckpoint file removed.")

print("\n" + "=" * 60)
print("GENERATION COMPLETE")
print("=" * 60)
print(f"\nArtifacts created:")
print(f"  1. {DATASET_PATH}")
print(f"  2. {SPLIT_PATH}")
print(f"  3. {QUALITY_REPORT_PATH}")

Dataset saved to: /content/drive/MyDrive/SeaAlert/data/processed/02seaalert.csv
Shape: (1872, 16)

Columns: ['idx', 'text', 'label', 'style', 'scenario_type', 'has_codeword', 'codeword', 'text_masked', 'vessel', 'call_sign', 'mmsi', 'location', 'weather', 'pob', 'nature', 'injury']

Checkpoint file removed.

GENERATION COMPLETE

Artifacts created:
  1. /content/drive/MyDrive/SeaAlert/data/processed/02seaalert.csv
  2. /content/drive/MyDrive/SeaAlert/results/split_indices.csv
  3. /content/drive/MyDrive/SeaAlert/results/data_quality_report.txt
